# Open-source LLM Text Classification Benchmark (local vLLM engine)

This notebook benchmarks binary text classification with open-source instruction LLMs by running vLLM directly in Python (`vllm.LLM`), with Vietnamese prompting modes across multiple shot levels:

- `shot_0`
- `shot_1`
- `shot_3`
- `shot_5`
- `shot_10`
- `shot_15`
- `shot_20`

Goals:

1. Keep data and evaluation logic compatible with the existing benchmark (`accuracy`, `macro_f1`).
2. Optimize for inference speed and VRAM efficiency (short outputs, capped input length, tuned batch/runtime profiles).
3. Export benchmark summaries to CSV and JSON.

In [ ]:
!pip install -qU huggingface_hub
!hf auth login --token <your_hf_token>

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.2 MB/s eta 0:00:00
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `public_token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `public_token`


In [2]:
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.2/248.2 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.7/748.7

In [3]:
!find / -name "libcuda.so*" 2>/dev/null

/root/.julia/artifacts/155d74984311b447417e2c4aaf2e35d918852a16/lib/libcuda.so.1
/root/.julia/artifacts/155d74984311b447417e2c4aaf2e35d918852a16/lib/libcuda.so.590.48.01
/root/.julia/artifacts/155d74984311b447417e2c4aaf2e35d918852a16/lib/libcuda.so
/usr/local/nvidia/lib64/libcuda.so
/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/nvidia/lib64/libcuda.so.580.105.08
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/compat/libcuda.so.570.124.06


In [4]:
!sudo ln -s /usr/local/nvidia/lib64/libcuda.so.1 /usr/lib/x86_64-linux-gnu/libcuda.so

In [5]:
from __future__ import annotations

import json
import re
import time
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, precision_score, recall_score
from tqdm import tqdm

try:
    from vllm import LLM, SamplingParams
except Exception as exc:
    raise RuntimeError(
        "Please install vLLM first: pip install -U vllm"
    ) from exc

In [6]:
# SHOT_VALUES: tuple[int, ...] = (0, 1, 3, 5,)
SHOT_VALUES: tuple[int, ...] = (10, 15, 20)


def mode_name_from_shot(shot: int) -> str:
    return f"shot_{int(shot)}"


def shot_count_from_mode(mode: str) -> int:
    if not mode.startswith("shot_"):
        raise ValueError(f"Unsupported prompt mode '{mode}'. Expected format: shot_<k>")
    try:
        shot = int(mode.split("_", 1)[1])
    except Exception as exc:
        raise ValueError(f"Invalid prompt mode '{mode}'. Expected format: shot_<k>") from exc
    if shot < 0:
        raise ValueError(f"Shot count must be >= 0, got {shot} from mode '{mode}'")
    return shot


DEFAULT_VI_EXAMPLE_POOL: list[dict[str, Any]] = [
    {"text": "Giá điện bình quân đã tăng 4,8% từ tháng 11/2023 theo quyết định của Bộ Công Thương.", "label": 1},
    {"text": "Tôi thấy bộ phim này quá hay và đáng xem nhất năm.", "label": 0},
    {"text": "Tỷ lệ thất nghiệp quý I/2025 của Việt Nam là 2,2%.", "label": 1},
    {"text": "Theo mình thì ăn tối sau 8 giờ là không tốt.", "label": 0},
    {"text": "Bài đăng nói rằng hơn 20.000 cây xanh đã bị chặt trong một tháng tại thành phố.", "label": 1},
    {"text": "Bạn nên đọc cuốn sách này vì nó truyền cảm hứng mạnh mẽ.", "label": 0},
    {"text": "Phát biểu cho rằng vaccine X làm tăng nguy cơ đột quỵ gấp 3 lần.", "label": 1},
    {"text": "Tôi rất lo lắng về tình hình giao thông hiện nay.", "label": 0},
    {"text": "Doanh nghiệp A báo lỗ 1.200 tỷ đồng trong báo cáo tài chính năm 2024.", "label": 1},
    {"text": "Chính sách mới này có vẻ không công bằng với người trẻ.", "label": 0},
    {"text": "Bài viết khẳng định nước mắm công nghiệp chứa thạch tín vượt ngưỡng an toàn.", "label": 1},
    {"text": "Món ăn này ngon hơn hẳn quán bên cạnh.", "label": 0},
    {"text": "Có thông tin cho rằng 70% học sinh tiểu học bị cận thị ở thành phố B.", "label": 1},
    {"text": "Tôi nghĩ đội bóng này sẽ vô địch vì họ có tinh thần tốt.", "label": 0},
    {"text": "Bản tin nêu cầu C sẽ hoàn thành trước tiến độ 6 tháng.", "label": 1},
    {"text": "Thời tiết hôm nay thật khó chịu.", "label": 0},
    {"text": "Tuyên bố rằng uống nước chanh chữa khỏi ung thư.", "label": 1},
    {"text": "Theo cảm nhận cá nhân, khu phố này an toàn hơn trước.", "label": 0},
    {"text": "Bài đăng nói thu nhập bình quân đầu người của tỉnh D đã vượt 100 triệu đồng/năm.", "label": 1},
    {"text": "Mình thích làm việc ở nhà hơn vì thấy thoải mái.", "label": 0},
]

PROMPT_TEMPLATES: dict[str, dict[str, Any]] = {
    "vi": {
        "system": (
            "Bạn là bộ phân loại nhị phân cho nhiệm vụ check-worthiness detection. "
            "Mục tiêu là xác định phát biểu có cần được kiểm chứng thực tế hay không. "
            "Nhãn 1: phát biểu chứa mệnh đề thực tế có thể kiểm tra đúng/sai bằng nguồn đáng tin cậy. "
            "Nhãn 0: ý kiến, cảm xúc, lời khuyên, câu hỏi, hoặc nội dung quá mơ hồ/không thể kiểm chứng. "
            "Không đánh giá phát biểu đúng hay sai; chỉ đánh giá mức độ cần kiểm chứng."
        ),
        "task_prefix": (
            "Nhiệm vụ: phân loại phát biểu đầu vào thành 0 hoặc 1. "
            "Nếu có ít nhất một mệnh đề thực tế, cụ thể, có thể kiểm chứng thì chọn 1; ngược lại chọn 0."
        ),
        "output_rule": "Chỉ trả về đúng một token duy nhất: 0 hoặc 1. Không thêm giải thích.",
        "examples_title": "Ví dụ tham chiếu:",
        "input_prefix": "Phát biểu",
        "label_suffix": "Nhãn",
        "mode_instructions": {
            mode_name_from_shot(shot): (
                "Không sử dụng ví dụ tham chiếu."
                if shot == 0
                else f"Sử dụng đúng {shot} ví dụ tham chiếu để suy luận nội bộ trước khi phân loại."
            )
            for shot in SHOT_VALUES
        },
    }
}

@dataclass
class CFG:
    data_path: str = "/kaggle/input/datasets/giapkim/check-worthiness-test/test.jsonl"
    text_key: str = "text"
    label_key: str = "label"
    perplexity_key: str = "perplexity_score"
    seed: int = 42
    sample_limit: int | None = None

    model_names: tuple[str, ...] = (
        # tiny
        # "Qwen/Qwen2.5-1.5B-Instruct",
        # "meta-llama/Llama-3.2-1B-Instruct",
        "microsoft/Phi-3.5-mini-instruct",
        
        # small
        # "meta-llama/Llama-3.1-8B-Instruct",
        # "Qwen/Qwen2.5-7B-Instruct",
        # "mistralai/Mistral-7B-Instruct-v0.3",
        # "google/gemma-2-9b-it",
        # "vinai/PhoGPT-7B5-Instruct",
    )
    profile_name: str = "balanced"
    profiles: dict[str, dict[str, Any]] = field(
        default_factory=lambda: {
            "fast": {
                "tensor_parallel_size": 2,
                "dtype": "float16", # float32 for gemma 2
                "gpu_memory_utilization": 0.88,
                "max_model_len": 2048,
                "max_num_seqs": 32,
            },
            "balanced": {
                "tensor_parallel_size": 2,
                "dtype": "float16", # float32 for gemma 2
                "gpu_memory_utilization": 0.9,
                "max_model_len": 2048,
                "max_num_seqs": 32,
            },
        }
    )
    runtime_overrides: dict[str, Any] = field(default_factory=dict)
    trust_remote_code: bool = True

    prompt_language: str = "vi"
    prompt_modes: tuple[str, ...] = tuple(mode_name_from_shot(shot) for shot in SHOT_VALUES)
    exemplar_max_chars: int = 240
    max_input_chars: int = 1400
    user_examples: dict[str, list[dict[str, Any]]] = field(
        default_factory=lambda: {
            mode_name_from_shot(shot): [{**ex} for ex in DEFAULT_VI_EXAMPLE_POOL[:shot]]
            for shot in SHOT_VALUES
        }
    )

    quick_run: bool = False
    quick_max_samples: int = 100

    temperature: float = 0.0
    top_p: float = 1.0
    top_k: int = -1
    max_tokens: int = 4
    max_retries: int = 3
    retry_base_sleep_s: float = 1.0
    batch_size: int = 64

    length_bin_count: int = 10
    ppl_bin_count: int = 10

    output_root: str = "results_open_source_llm_vllm"


def resolve_runtime(cfg: CFG) -> dict[str, Any]:
    if cfg.profile_name not in cfg.profiles:
        choices = ", ".join(sorted(cfg.profiles))
        raise ValueError(f"Unknown profile '{cfg.profile_name}'. Available: {choices}")
    if cfg.prompt_language != "vi":
        raise ValueError("Only Vietnamese prompt template is supported. Set cfg.prompt_language = 'vi'.")

    mode_instructions = PROMPT_TEMPLATES["vi"]["mode_instructions"]
    for mode in cfg.prompt_modes:
        shot_count_from_mode(mode)
        if mode not in mode_instructions:
            raise ValueError(f"Missing prompt instruction for mode '{mode}'")

    runtime = dict(cfg.profiles[cfg.profile_name])
    runtime.update(cfg.runtime_overrides)
    return runtime


cfg = CFG()
runtime_cfg = resolve_runtime(cfg)
np.random.seed(cfg.seed)
print(
    {
        "model_names": list(cfg.model_names),
        "prompt_language": cfg.prompt_language,
        "prompt_modes": list(cfg.prompt_modes),
        "profile_name": cfg.profile_name,
        "perplexity_key": cfg.perplexity_key,
        "length_bin_count": cfg.length_bin_count,
        "ppl_bin_count": cfg.ppl_bin_count,
        **runtime_cfg,
    }
)

{'model_names': ['microsoft/Phi-3.5-mini-instruct'], 'prompt_language': 'vi', 'prompt_modes': ['shot_10', 'shot_15', 'shot_20'], 'profile_name': 'balanced', 'perplexity_key': 'perplexity_score', 'length_bin_count': 10, 'ppl_bin_count': 10, 'tensor_parallel_size': 2, 'dtype': 'float16', 'gpu_memory_utilization': 0.9, 'max_model_len': 2048, 'max_num_seqs': 32}


In [7]:
_BASIC_PUNCTUATION = set(".,;:!?\"'()[]{}-_/\\")

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u200b", " ").replace("\ufeff", " ")

    filtered_chars = []
    for ch in text:
        if ch.isspace():
            filtered_chars.append(" ")
            continue

        category = unicodedata.category(ch)
        if category[0] in {"L", "N"} or ch in _BASIC_PUNCTUATION:
            filtered_chars.append(ch)

    text = "".join(filtered_chars)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_jsonl_robust(
    path: str,
    text_key: str,
    label_key: str,
    perplexity_key: str,
    quick_max_samples: int = 10000,
) -> tuple[pd.DataFrame, dict[str, int]]:
    rows: list[dict[str, Any]] = []
    stats = {
        "total_lines": 0,
        "empty_lines": 0,
        "malformed_json": 0,
        "empty_text": 0,
        "invalid_label": 0,
        "invalid_perplexity": 0,
        "valid_rows": 0,
    }

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            stats["total_lines"] += 1
            line = line.strip()
            if not line:
                stats["empty_lines"] += 1
                continue

            try:
                obj = json.loads(line)
            except Exception:
                stats["malformed_json"] += 1
                continue

            txt = clean_text(obj.get(text_key, ""))
            if not txt:
                stats["empty_text"] += 1
                continue

            try:
                lbl = int(obj.get(label_key, -1))
            except Exception:
                stats["invalid_label"] += 1
                continue

            if lbl not in (0, 1):
                stats["invalid_label"] += 1
                continue

            try:
                ppl = float(obj.get(perplexity_key))
            except Exception:
                stats["invalid_perplexity"] += 1
                continue

            if not np.isfinite(ppl):
                stats["invalid_perplexity"] += 1
                continue

            rows.append(
                {
                    "text": txt,
                    "label": lbl,
                    "source_line": line_no,
                    "text_len_ws": int(len(txt.split())),
                    "perplexity_score": float(ppl),
                }
            )

    df = pd.DataFrame(rows)
    stats["valid_rows"] = int(len(df))
    stats["dropped_rows"] = int(
        stats["empty_lines"]
        + stats["malformed_json"]
        + stats["empty_text"]
        + stats["invalid_label"]
        + stats["invalid_perplexity"]
    )

    if df.empty:
        raise ValueError(f"No valid rows found in {path}. stats={stats}")

    print("Data loading summary:")
    print(f" - total_lines: {stats['total_lines']:,}")
    print(f" - valid_rows: {stats['valid_rows']:,}")
    print(f" - dropped_rows: {stats['dropped_rows']:,}")
    print(f"   - empty_lines: {stats['empty_lines']:,}")
    print(f"   - malformed_json: {stats['malformed_json']:,}")
    print(f"   - empty_text: {stats['empty_text']:,}")
    print(f"   - invalid_label: {stats['invalid_label']:,}")
    print(f"   - invalid_perplexity: {stats['invalid_perplexity']:,}")

    if CFG.quick_run:
        df = df.iloc[:quick_max_samples, :]
    return df.reset_index(drop=True), stats


eval_df, data_load_stats = load_jsonl_robust(
    cfg.data_path,
    cfg.text_key,
    cfg.label_key,
    cfg.perplexity_key,
    cfg.quick_max_samples,
)
if cfg.sample_limit is not None:
    eval_df = eval_df.sample(n=min(cfg.sample_limit, len(eval_df)), random_state=cfg.seed).reset_index(drop=True)
    data_load_stats["sample_limit_applied"] = int(cfg.sample_limit)
else:
    data_load_stats["sample_limit_applied"] = None

print(f"Evaluation rows: {len(eval_df):,}")

Data loading summary:
 - total_lines: 120,622
 - valid_rows: 120,622
 - dropped_rows: 0
   - empty_lines: 0
   - malformed_json: 0
   - empty_text: 0
   - invalid_label: 0
   - invalid_perplexity: 0
Evaluation rows: 120,622


In [8]:
def truncate_for_prompt(text: str, max_chars: int) -> str:
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3].rstrip() + "..."


def normalize_user_examples(
    user_examples: dict[str, list[dict[str, Any]]],
    max_chars: int,
    prompt_modes: tuple[str, ...],
) -> dict[str, list[dict[str, int | str]]]:
    normalized: dict[str, list[dict[str, int | str]]] = {mode: [] for mode in prompt_modes}

    for mode in prompt_modes:
        target_shot = shot_count_from_mode(mode)
        if target_shot == 0:
            continue
        for ex in user_examples.get(mode, []):
            try:
                text = truncate_for_prompt(str(ex.get("text", "")), max_chars)
                label = int(ex.get("label", -1))
            except Exception:
                continue
            if text and label in (0, 1):
                normalized[mode].append({"text": text, "label": label})

        if len(normalized[mode]) > target_shot:
            normalized[mode] = normalized[mode][:target_shot]

    return normalized


def format_examples(exemplars: list[dict[str, int | str]], title: str) -> str:
    if not exemplars:
        return ""
    lines = [title]
    for i, ex in enumerate(exemplars, start=1):
        lines.append(f"[{i}] text: {ex['text']}")
        lines.append(f"[{i}] label: {ex['label']}")
    return "\n".join(lines)


def build_messages(
    text: str,
    mode: str,
    prompt_language: str,
    templates: dict[str, dict[str, Any]],
    exemplars_by_mode: dict[str, list[dict[str, int | str]]],
    max_input_chars: int,
):
    if prompt_language not in templates:
        raise ValueError(f"Unsupported prompt language: {prompt_language}")

    prompt_cfg = templates[prompt_language]
    input_text = truncate_for_prompt(text, max_input_chars)
    exemplars = exemplars_by_mode.get(mode, [])

    user_lines = [
        prompt_cfg["task_prefix"],
        prompt_cfg["mode_instructions"][mode],
        prompt_cfg["output_rule"],
    ]

    examples_block = format_examples(exemplars, prompt_cfg["examples_title"])
    if examples_block:
        user_lines.extend(["", examples_block])

    user_lines.extend(["", f"{prompt_cfg['input_prefix']}: {input_text}", f"{prompt_cfg['label_suffix']}: "])
    user_prompt = "\n".join(user_lines)

    return [
        {"role": "system", "content": prompt_cfg["system"]},
        {"role": "user", "content": user_prompt},
    ]


exemplars_by_mode = normalize_user_examples(cfg.user_examples, cfg.exemplar_max_chars, cfg.prompt_modes)
for mode in cfg.prompt_modes:
    expected = shot_count_from_mode(mode)
    actual = len(exemplars_by_mode.get(mode, []))
    print(mode, "num_user_examples=", actual, "expected=", expected)

shot_10 num_user_examples= 10 expected= 10
shot_15 num_user_examples= 15 expected= 15
shot_20 num_user_examples= 20 expected= 20


In [9]:
LABEL_RE = re.compile(r"\b([01])\b")


def parse_label(raw_text: str) -> int | None:
    text = (raw_text or "").strip().lower()
    m = LABEL_RE.search(text)
    if m:
        return int(m.group(1))
    if text in {"0", "1"}:
        return int(text)

    neg_markers = [
        "label 0",
        "khong can kiem chung",
        "không cần kiểm chứng",
        "khong check-worthy",
        "not check-worthy",
        "opinion",
        "y kien",
    ]
    pos_markers = [
        "label 1",
        "can kiem chung",
        "cần kiểm chứng",
        "check-worthy",
        "factual claim",
        "co the kiem chung",
        "có thể kiểm chứng",
    ]

    if any(x in text for x in neg_markers):
        return 0
    if any(x in text for x in pos_markers):
        return 1
    return None


def build_llm_engine(model_name: str, cfg: CFG, runtime_cfg: dict[str, Any]) -> LLM:
    return LLM(
        model=model_name,
        tensor_parallel_size=int(runtime_cfg["tensor_parallel_size"]),
        dtype=str(runtime_cfg["dtype"]),
        gpu_memory_utilization=float(runtime_cfg["gpu_memory_utilization"]),
        max_model_len=int(runtime_cfg["max_model_len"]),
        max_num_seqs=int(runtime_cfg["max_num_seqs"]),
        trust_remote_code=cfg.trust_remote_code,
        seed=cfg.seed,
    )


def build_sampling_params(cfg: CFG) -> SamplingParams:
    return SamplingParams(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=cfg.max_tokens,
        top_k=cfg.top_k,
        seed=cfg.seed,
    )


def render_chat_prompt(llm: LLM, messages: list[dict[str, str]]) -> str:
    tokenizer = llm.get_tokenizer()
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    # Fallback for tokenizers without chat template.
    chunks = []
    for msg in messages:
        role = str(msg.get("role", "user")).strip().upper()
        content = str(msg.get("content", "")).strip()
        chunks.append(f"{role}: {content}")
    chunks.append("ASSISTANT:")
    return "\n\n".join(chunks)


def call_local_vllm_batch_with_retry(
    llm: LLM,
    prompts: list[str],
    sampling_params: SamplingParams,
    cfg: CFG,
) -> list[dict[str, Any]]:
    err: str | None = None
    for attempt in range(cfg.max_retries + 1):
        try:
            outs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=False)
            rows: list[dict[str, Any]] = []
            for out in outs:
                best = out.outputs[0] if out.outputs else None
                rows.append(
                    {
                        "ok": True,
                        "raw": (best.text if best else "") or "",
                        "prompt_tokens": len(getattr(out, "prompt_token_ids", []) or []),
                        "completion_tokens": len(getattr(best, "token_ids", []) or []) if best else 0,
                        "error": None,
                    }
                )
            return rows
        except Exception as exc:
            err = str(exc)
            if attempt >= cfg.max_retries:
                break
            sleep_s = cfg.retry_base_sleep_s * (2 ** attempt)
            time.sleep(sleep_s)

    return [
        {
            "ok": False,
            "raw": "",
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "error": err,
        }
        for _ in prompts
    ]


def batched_indices(n: int, bs: int):
    for i in range(0, n, bs):
        yield i, min(i + bs, n)


def run_mode_inference(
    eval_df: pd.DataFrame,
    llm: LLM,
    model_name: str,
    mode: str,
    exemplars_by_mode: dict[str, list[dict[str, int | str]]],
    sampling_params: SamplingParams,
    cfg: CFG,
) -> tuple[pd.DataFrame, float]:
    records: list[dict[str, Any]] = []
    t0_all = time.perf_counter()

    pbar = tqdm(batched_indices(len(eval_df), cfg.batch_size), desc="Evaluating")
    for s, e in pbar:
        batch = eval_df.iloc[s:e]
        prompts = [
            render_chat_prompt(
                llm,
                build_messages(
                    row.text,
                    mode,
                    cfg.prompt_language,
                    PROMPT_TEMPLATES,
                    exemplars_by_mode,
                    cfg.max_input_chars,
                ),
            )
            for row in batch.itertuples(index=False)
        ]

        t0_batch = time.perf_counter()
        out_rows = call_local_vllm_batch_with_retry(llm, prompts, sampling_params, cfg)
        batch_latency_s = time.perf_counter() - t0_batch
        per_sample_latency_s = batch_latency_s / len(batch) if len(batch) else 0.0

        for row, out in zip(batch.itertuples(index=False), out_rows):
            pred = parse_label(out["raw"])
            records.append(
                {
                    "text": row.text,
                    "label": int(row.label),
                    "pred_label": pred,
                    "raw_response": out["raw"],
                    "latency_s": per_sample_latency_s,
                    "prompt_tokens": out["prompt_tokens"],
                    "completion_tokens": out["completion_tokens"],
                    "error": out["error"],
                    "model_name": model_name,
                    "source_line": int(row.source_line),
                    "text_len_ws": int(row.text_len_ws),
                    "perplexity_score": float(row.perplexity_score),
                }
            )

    elapsed = time.perf_counter() - t0_all
    out_df = pd.DataFrame(records)
    return out_df, elapsed

In [10]:
IMPORTANT_METRIC_KEYS = [
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "precision",
    "recall",
    "f1",
    "precision_neg",
    "recall_neg",
    "f1_neg",
    "support_neg",
    "precision_pos",
    "recall_pos",
    "f1_pos",
    "support_pos",
]


def compute_basic_metrics(y_true: pd.Series | np.ndarray, y_pred: pd.Series | np.ndarray) -> dict[str, float]:
    y_true_np = np.asarray(y_true, dtype=int)
    y_pred_np = np.asarray(y_pred, dtype=int)

    prec, rec, f1_cls, support = precision_recall_fscore_support(
        y_true_np,
        y_pred_np,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy_score(y_true_np, y_pred_np)),
        "macro_f1": float(f1_score(y_true_np, y_pred_np, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true_np, y_pred_np, average="weighted", zero_division=0)),
        "precision": float(precision_score(y_true_np, y_pred_np, zero_division=0)),
        "recall": float(recall_score(y_true_np, y_pred_np, zero_division=0)),
        "f1": float(f1_score(y_true_np, y_pred_np, zero_division=0)),
        "precision_neg": float(prec[0]),
        "recall_neg": float(rec[0]),
        "f1_neg": float(f1_cls[0]),
        "support_neg": int(support[0]),
        "precision_pos": float(prec[1]),
        "recall_pos": float(rec[1]),
        "f1_pos": float(f1_cls[1]),
        "support_pos": int(support[1]),
    }


def compute_classification_metrics(df_pred: pd.DataFrame) -> dict[str, float]:
    valid_mask = df_pred["pred_label"].isin([0, 1])
    coverage = float(valid_mask.mean()) if len(df_pred) else 0.0
    if valid_mask.sum() == 0:
        out = {k: float("nan") for k in IMPORTANT_METRIC_KEYS}
        out["valid_coverage"] = coverage
        return out

    y_true = df_pred.loc[valid_mask, "label"].astype(int)
    y_pred = df_pred.loc[valid_mask, "pred_label"].astype(int)
    out = compute_basic_metrics(y_true, y_pred)
    out["valid_coverage"] = coverage
    return out


def compute_quantile_edges(values: pd.Series | np.ndarray, n_bins: int) -> np.ndarray:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.array([0.0, 1.0], dtype=float)

    if n_bins < 1:
        n_bins = 1

    edges = np.quantile(vals, np.linspace(0.0, 1.0, n_bins + 1))
    edges = np.unique(edges)
    if edges.size < 2:
        x = float(edges[0])
        return np.array([x - 1.0, x + 1.0], dtype=float)
    return edges.astype(float)


def compute_bin_metrics(
    df_pred: pd.DataFrame,
    value_col: str,
    edges: np.ndarray,
    prefix: str,
) -> dict[str, float]:
    values = pd.to_numeric(df_pred[value_col], errors="coerce").to_numpy(dtype=float)
    finite_mask = np.isfinite(values)
    valid_pred_mask = df_pred["pred_label"].isin([0, 1]).to_numpy()
    bin_ids = np.digitize(values, edges[1:-1], right=False)

    out: dict[str, float] = {}
    n_bins = len(edges) - 1
    for b in range(n_bins):
        key = f"{prefix}_bin_{b}"
        in_bin = (bin_ids == b) & finite_mask
        support_total = int(in_bin.sum())

        out[f"{key}/edge_low"] = float(edges[b])
        out[f"{key}/edge_high"] = float(edges[b + 1])
        out[f"{key}/support_total"] = support_total

        valid_in_bin = in_bin & valid_pred_mask
        out[f"{key}/valid_coverage"] = float(valid_in_bin.sum() / support_total) if support_total else float("nan")

        if valid_in_bin.sum() == 0:
            for metric_key in IMPORTANT_METRIC_KEYS:
                out[f"{key}/{metric_key}"] = float("nan")
            continue

        y_true = df_pred.loc[valid_in_bin, "label"].astype(int)
        y_pred = df_pred.loc[valid_in_bin, "pred_label"].astype(int)
        sub_metrics = compute_basic_metrics(y_true, y_pred)
        for metric_key, metric_val in sub_metrics.items():
            out[f"{key}/{metric_key}"] = metric_val

    return out


def build_mode_summary(df_pred: pd.DataFrame, runtime_metrics: dict[str, float], cfg: CFG) -> dict[str, Any]:
    cls_metrics = compute_classification_metrics(df_pred)
    len_edges = compute_quantile_edges(df_pred["text_len_ws"], cfg.length_bin_count)
    ppl_edges = compute_quantile_edges(df_pred["perplexity_score"], cfg.ppl_bin_count)

    bin_metrics = {
        **compute_bin_metrics(df_pred, "text_len_ws", len_edges, "len"),
        **compute_bin_metrics(df_pred, "perplexity_score", ppl_edges, "ppl"),
    }

    return {
        "overall_metrics": {k: cls_metrics[k] for k in ["accuracy", "macro_f1", "weighted_f1", "precision", "recall", "f1", "valid_coverage"]},
        "class_metrics": {
            k: cls_metrics[k]
            for k in [
                "precision_neg",
                "recall_neg",
                "f1_neg",
                "support_neg",
                "precision_pos",
                "recall_pos",
                "f1_pos",
                "support_pos",
            ]
        },
        "runtime_metrics": runtime_metrics,
        "bin_metrics": bin_metrics,
        "bin_edges": {
            "len": [float(x) for x in len_edges],
            "ppl": [float(x) for x in ppl_edges],
        },
    }


def summarize_runtime(df_pred: pd.DataFrame, elapsed_s: float) -> dict[str, float]:
    n = len(df_pred)
    total_prompt_tokens = int(df_pred["prompt_tokens"].sum())
    total_completion_tokens = int(df_pred["completion_tokens"].sum())
    total_tokens = total_prompt_tokens + total_completion_tokens
    parse_fail_rate = float((~df_pred["pred_label"].isin([0, 1])).mean()) if n else 0.0
    error_rate = float(df_pred["error"].notna().mean()) if n else 0.0
    return {
        "num_samples": n,
        "elapsed_s": float(elapsed_s),
        "avg_latency_s": float(df_pred["latency_s"].mean()) if n else 0.0,
        "samples_per_second": float(n / elapsed_s) if elapsed_s > 0 else 0.0,
        "tokens_per_second": float(total_tokens / elapsed_s) if elapsed_s > 0 else 0.0,
        "prompt_tokens": total_prompt_tokens,
        "completion_tokens": total_completion_tokens,
        "parse_fail_rate": parse_fail_rate,
        "error_rate": error_rate,
    }


def sanitize_model_name(model_name: str) -> str:
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", model_name).strip("._")
    return safe or "model"

In [11]:
eval_df = eval_df.reset_index(drop=True)

print("=" * 80)
print(f"Models: {list(cfg.model_names)}")
print(f"Profile: {cfg.profile_name}")
print(f"Prompt language: {cfg.prompt_language}")
print("Runtime:", runtime_cfg)
print("Data quality stats:", data_load_stats)

all_runs: list[dict[str, Any]] = []
all_predictions: list[pd.DataFrame] = []
model_summaries: dict[str, dict[str, Any]] = {}
sampling_params = build_sampling_params(cfg)

for model_name in cfg.model_names:
    print("=" * 80)
    print(f"Loading model: {model_name}")
    llm = build_llm_engine(model_name, cfg, runtime_cfg)

    model_summary: dict[str, Any] = {
        "model_name": model_name,
        "profile_name": cfg.profile_name,
        "prompt_language": cfg.prompt_language,
        "binning": {
            "length_bin_count": int(cfg.length_bin_count),
            "ppl_bin_count": int(cfg.ppl_bin_count),
            "perplexity_key": cfg.perplexity_key,
        },
        "modes": {},
    }

    for mode in cfg.prompt_modes:
        print(f"Running model={model_name} mode={mode} on {len(eval_df):,} samples...")
        pred_df, elapsed_s = run_mode_inference(
            eval_df=eval_df,
            llm=llm,
            model_name=model_name,
            mode=mode,
            exemplars_by_mode=exemplars_by_mode,
            sampling_params=sampling_params,
            cfg=cfg,
        )

        metric = compute_classification_metrics(pred_df)
        runtime = summarize_runtime(pred_df, elapsed_s)
        run_row = {
            "model_name": model_name,
            "prompt_mode": mode,
            "profile_name": cfg.profile_name,
            "prompt_language": cfg.prompt_language,
            **runtime_cfg,
            **metric,
            **runtime,
        }
        all_runs.append(run_row)

        mode_summary = build_mode_summary(pred_df, runtime, cfg)
        model_summary["modes"][mode] = mode_summary

        pred_df["prompt_mode"] = mode
        pred_df["profile_name"] = cfg.profile_name
        pred_df["prompt_language"] = cfg.prompt_language
        all_predictions.append(pred_df)

        print(
            " -> "
            f"acc={run_row['accuracy']:.4f}, "
            f"macro_f1={run_row['macro_f1']:.4f}, "
            f"coverage={run_row['valid_coverage']:.3f}, "
            f"sps={run_row['samples_per_second']:.2f}, "
            f"parse_fail={run_row['parse_fail_rate']:.3f}"
        )

    model_summary["generated_at"] = datetime.now(timezone.utc).isoformat()
    model_summaries[model_name] = model_summary
    del llm

results_df = pd.DataFrame(all_runs)
results_df = results_df.sort_values(
    ["macro_f1", "accuracy", "samples_per_second"],
    ascending=[False, False, False],
).reset_index(drop=True)

predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
results_df

Models: ['microsoft/Phi-3.5-mini-instruct']
Profile: balanced
Prompt language: vi
Runtime: {'tensor_parallel_size': 2, 'dtype': 'float16', 'gpu_memory_utilization': 0.9, 'max_model_len': 2048, 'max_num_seqs': 32}
Data quality stats: {'total_lines': 120622, 'empty_lines': 0, 'malformed_json': 0, 'empty_text': 0, 'invalid_label': 0, 'invalid_perplexity': 0, 'valid_rows': 120622, 'dropped_rows': 0, 'sample_limit_applied': None}
Loading model: microsoft/Phi-3.5-mini-instruct
INFO 05-17 14:32:20 [utils.py:240] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'seed': 42, 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'microsoft/Phi-3.5-mini-instruct'}


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


INFO 05-17 14:32:39 [model.py:568] Resolved architecture: Phi3ForCausalLM
WARNING 05-17 14:32:39 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-17 14:32:39 [model.py:1697] Using max model len 2048
INFO 05-17 14:32:39 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-17 14:32:39 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-17 14:32:39 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


...

Evaluating: 1885it [3:19:27,  6.35s/it]


 -> acc=0.4652, macro_f1=0.4611, coverage=1.000, sps=10.08, parse_fail=0.000
Running model=microsoft/Phi-3.5-mini-instruct mode=shot_20 on 120,622 samples...


Evaluating: 1885it [3:54:35,  7.47s/it]


(EngineCore pid=184) INFO 05-18 00:47:18 [core.py:1242] Shutdown initiated (timeout=0)
(EngineCore pid=184) INFO 05-18 00:47:18 [core.py:1265] Shutdown complete
(Worker_TP0 pid=202) (Worker_TP1 pid=203) INFO 05-18 00:47:18 [multiproc_executor.py:775] Parent process exited, terminating worker queues
(Worker_TP0 pid=202) INFO 05-18 00:47:18 [multiproc_executor.py:872] WorkerProc shutting down.
INFO 05-18 00:47:18 [multiproc_executor.py:872] WorkerProc shutting down.
 -> acc=0.4750, macro_f1=0.4684, coverage=1.000, sps=8.57, parse_fail=0.000


,model_name,prompt_mode,profile_name,prompt_language,tensor_parallel_size,dtype,gpu_memory_utilization,max_model_len,max_num_seqs,accuracy,...,valid_coverage,num_samples,elapsed_s,avg_latency_s,samples_per_second,tokens_per_second,prompt_tokens,completion_tokens,parse_fail_rate,error_rate
0,microsoft/Phi-3.5-mini-instruct,shot_10,balanced,vi,2,float16,0.9,2048,32,0.532776,...,0.999992,120622,10622.482020,0.087790,11.355350,12714.559530,134698313,361867,0.000008,0.0
1,microsoft/Phi-3.5-mini-instruct,shot_20,balanced,vi,2,float16,0.9,2048,32,0.475033,...,0.999992,120622,14075.351092,0.116407,8.569733,14231.736224,199954815,361869,0.000008,0.0
2,microsoft/Phi-3.5-mini-instruct,shot_15,balanced,vi,2,float16,0.9,2048,32,0.465176,...,0.999992,120622,11967.403827,0.098938,10.079212,14097.771032,168351851,361868,0.000008,0.0


In [12]:
output_root = Path(cfg.output_root)
output_root.mkdir(parents=True, exist_ok=True)

summary_csv = output_root / "benchmark_summary.csv"
summary_json = output_root / "benchmark_summary.json"
predictions_csv = output_root / "benchmark_predictions.csv"
run_config_json = output_root / "run_config.json"

results_df.to_csv(summary_csv, index=False)
results_df.to_json(summary_json, orient="records", force_ascii=False, indent=2)
if not predictions_df.empty:
    predictions_df.to_csv(predictions_csv, index=False)

model_summary_files: list[Path] = []
for model_name, model_payload in model_summaries.items():
    safe_name = sanitize_model_name(model_name)
    model_summary_path = output_root / f"{safe_name}_summary.json"
    with open(model_summary_path, "w", encoding="utf-8") as f:
        json.dump(model_payload, f, ensure_ascii=False, indent=2)
    model_summary_files.append(model_summary_path)

run_config = {
    **cfg.__dict__,
    "prompt_templates": PROMPT_TEMPLATES,
    "active_runtime": runtime_cfg,
    "data_load_stats": data_load_stats,
}
with open(run_config_json, "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

print("Saved:")
print(" -", summary_csv)
print(" -", summary_json)
print(" -", predictions_csv)
print(" -", run_config_json)
for model_summary_path in model_summary_files:
    print(" -", model_summary_path)

Saved:
 - results_open_source_llm_vllm/benchmark_summary.csv
 - results_open_source_llm_vllm/benchmark_summary.json
 - results_open_source_llm_vllm/benchmark_predictions.csv
 - results_open_source_llm_vllm/run_config.json
 - results_open_source_llm_vllm/microsoft_Phi-3.5-mini-instruct_summary.json


## Local vLLM setup and research tuning (GPU 16-24GB)

### Install

```bash
pip install -U vllm
```

### Prompt objective (check-worthiness)

- Task: detect whether an input statement is **check-worthy**.
- Label definition:
  - `1`: contains at least one factual claim that can be verified with reliable evidence.
  - `0`: opinion/emotion/advice/question or unverifiable vague statement.
- Important: this is **not** truthfulness detection. We only classify whether the statement should be fact-checked.

### How to switch model/profile quickly

- Main knobs in `CFG`:
  - `cfg.model_names`: list models to benchmark in one run.
  - `cfg.profile_name`: pick runtime preset (`fast` or `balanced`).
  - `cfg.runtime_overrides`: override any preset field without editing the preset itself.
  - `cfg.prompt_language`: fixed to `"vi"` (Vietnamese only).
  - `cfg.prompt_modes`: choose among `shot_0`, `shot_1`, `shot_3`, `shot_5`, `shot_10`, `shot_15`, `shot_20`.
  - `cfg.user_examples`: user-authored Vietnamese check-worthiness examples, keyed by `shot_k` mode.

Example:

```python
cfg.model_names = (
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
)
cfg.profile_name = "fast"
cfg.prompt_modes = ("shot_0", "shot_5", "shot_10")
cfg.runtime_overrides = {
    "max_num_seqs": 48,
    "gpu_memory_utilization": 0.9,
}
cfg.user_examples["shot_5"] = [
    {"text": "Tỷ lệ lạm phát năm 2025 của Việt Nam vượt 6%.", "label": 1},
    {"text": "Mình thấy chính sách này khá hợp lý.", "label": 0},
    {"text": "Doanh nghiệp X công bố doanh thu quý II tăng 35%.", "label": 1},
    {"text": "Tôi không thích cách truyền thông đưa tin vụ này.", "label": 0},
    {"text": "Bài đăng nói tỉnh Y đã xóa 100% nhà tạm vào cuối năm 2024.", "label": 1},
]
runtime_cfg = resolve_runtime(cfg)
```

### Preset intent

- `fast`: higher throughput, lower VRAM pressure.
- `balanced`: more conservative throughput for stronger quality stability on larger models.

### Notebook-side tuning hints

- Keep `cfg.max_tokens` very small (2-4) for classification.
- Increase `cfg.batch_size` gradually and monitor `error_rate` / `parse_fail_rate`.
- Keep `cfg.max_input_chars` conservative to reduce prefill latency.
- Use `shot_0` as the speed baseline, then compare quality/latency changes across higher shot settings.
- For higher shot modes, maintain label balance in exemplars (roughly 50/50) and diversify domains (health, economy, education, policy).